In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# Load dataset
df_test = pd.read_csv("/kaggle/input/ana-erisk/df_testERisk19 (1).csv")  
df_train = pd.read_csv("/kaggle/input/ana-erisk/df_trainERisk19 (1).csv") 
df_val = pd.read_csv("/kaggle/input/ana-erisk/df_valERisk19 (1).csv") 

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570509 entries, 0 to 570508
Data columns (total 6 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   ID      570509 non-null  object
 1   TITLE   570509 non-null  object
 2   DATE    570509 non-null  object
 3   INFO    570509 non-null  object
 4   TEXT    570509 non-null  object
 5   label   570509 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 26.1+ MB


In [4]:
df_test_copy = df_test.copy()  
df_train_copy = df_train.copy() 
df_val_copy = df_val.copy() 

In [5]:
def merge_title_text(df):
    df["TITLE"] = df["TITLE"].fillna("")
    df["TEXT"] = df["TEXT"].fillna("")
    df["TEXT_MERGED"] = df["TITLE"] + " " + df["TEXT"]
    return df

In [6]:
df_test  = merge_title_text(df_test)

In [7]:
df_train  = merge_title_text(df_train)

In [8]:
df_val  = merge_title_text(df_val)

In [9]:
df_train[["TEXT_MERGED",'label']].head()

,TEXT_MERGED,label
0,"I don't know really, when I started to go...",1
1,"Dude, I think I know myself a lot better ...",1
2,"Yeah, obviously. The world doesn't look t...",1
3,"By supporting that law, are you saying th...",1
4,Nah I rather not.,1


In [10]:
text_col="TEXT_MERGED"
label_col='label'

In [11]:
pip install transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 76.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.

In [12]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch


def load_translator(src, tgt):
    model_name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return tokenizer, model


# Load models
tok_en_de, model_en_de = load_translator("en", "de")
tok_de_en, model_de_en = load_translator("de", "en")

# Device selection
device = "cuda" if torch.cuda.is_available() else "cpu"
model_en_de.to(device)
model_de_en.to(device)


def translate(texts, tok, model, device=device):
    batch = tok(texts, return_tensors="pt", padding=True, truncation=True)
    batch = {k: v.to(device) for k, v in batch.items()}  # FIX added
    out = model.generate(**batch)
    return tok.batch_decode(out, skip_special_tokens=True)


def back_translate(texts):
    de = translate(texts, tok_en_de, model_en_de)
    en = translate(de, tok_de_en, model_de_en)
    return en


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [13]:
qw = "Dude, I think I know myself..."
print(back_translate([qw]))

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

['Dude, I think I know myself...']


In [14]:
qw="""Freya Deco Moulded Plunge bra in Charcoal Grey. I bought the wrong size and returning internationally is too expensive for me. Listed on amazon at $48.50. Selling for $35 including shipping.

[Pictures](http://imgur.com/8l1YZA4)

[Amazon link](http://www.amazon.com/Freya-Womens-Moulded-Plunge-Black/dp/B002RVB9A0/ref=sr_1_1?ie=UTF8 qid=1429306099 sr=8-1"""
print(back_translate([qw]))

['Freya Deco Formed Plunge Bra in Charcoal Grey. I bought the wrong size and return internationally is too expensive for me. On Amazon at $48.50. Sale for $35 including shipping. [Images](http://imgur.com/8l1YZA4) [Amazon link](http://www.amazon.com/Freya-Womens-Moulded-Plunge-Black/dp/B002RVB9A0/ref=sr_1_1?ie=UTF8 qid=1429306099 sr=8-1']


In [15]:
#PRIMARY PREPROCESSING TASK FOR ALL DATAFRAMES

In [16]:
import re
import emoji

def preprocess_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Remove URLs
    #text = epreprocessoji.replace_emoji(text, replace='')         # Remove emojis (or use labels)
    text = re.sub(r'<.*?>', '', text)                    # Remove HTML tags
    text = re.sub(r'\s+', ' ', text).strip()             # Normalize whitespace
    return text

In [17]:
df_train[text_col] = df_train[text_col].apply(preprocess_text)

In [18]:
df_val[text_col] = df_val[text_col].apply(preprocess_text)

In [19]:
df_test[text_col] = df_test[text_col].apply(preprocess_text)

In [20]:
df_train[text_col]

0        I don't know really, when I started to go thro...
1        Dude, I think I know myself a lot better than ...
2        Yeah, obviously. The world doesn't look too ki...
3        By supporting that law, are you saying that me...
4                                        Nah I rather not.
                               ...                        
84829    So many people will die of natural causes the ...
84830    By the time this is built all cars will be sel...
84831                               Here we go ....maybe ?
84832                          Credit cards in the scanner
84833    Humanity works on a constant appropriation of ...
Name: TEXT_MERGED, Length: 84834, dtype: object

In [21]:
#### BALANCE USDERS ####

In [22]:
import random

def back_translate_fr(text):
    return text + " (fr aug)"

def back_translate_de(text):
    return text + " (de aug)"

def back_translate_es(text):
    return text + " (es aug)"

def paraphrase_t5(text):
    return text + " (t5 aug)"

def paraphrase_pegasus(text):
    return text + " (peg aug)"

def simple_synonym(text):
    return text + " (syn aug)"

# ---- Augmentation Pool ----
AUGMENTERS = [
    back_translate_fr,
    back_translate_de,
    back_translate_es,
    paraphrase_t5,
    paraphrase_pegasus,
    simple_synonym
]


def augment_text(text):
    """Randomly select one augmentation technique per text."""
    aug_method = random.choice(AUGMENTERS)
    return aug_method(text)


In [23]:
# ---------------------------
# Function: Balance Users by Synthetic User Generation
# ---------------------------
def balance_users(df: pd.DataFrame, label_col="label", id_col="ID"):
    # 1. Identify users and distribution
    user_counts = df.groupby(id_col)[label_col].first()
    
    positive_users = user_counts[user_counts == 1].index.tolist()
    negative_users = user_counts[user_counts == 0].index.tolist()

    print(f"🟢 Positive users: {len(positive_users)}")
    print(f"🔴 Negative users: {len(negative_users)}")

    # Determine minority and majority
    if len(positive_users) < len(negative_users):
        minority_users = positive_users
        minority_label = 1
        target_count = len(negative_users)
    else:
        minority_users = negative_users
        minority_label = 0
        target_count = len(positive_users)

    existing_ids = set(df[id_col])
    synthetic_users = []
    synthetic_user_map = {}  # mapping original -> count suffix

    print(f"📌 Target user count: {target_count}")

    # Shuffle for random sampling
    random.shuffle(minority_users)

    i = 0
    while len(minority_users) + len(synthetic_users) < target_count:
        source_user = random.choice(minority_users)
        source_posts = df[df[id_col] == source_user]

        # Generate new synthetic user ID
        user_base = source_user.replace("_S", "") if "_S" in source_user else source_user
        
        synthetic_index = synthetic_user_map.get(user_base, 0) + 1
        new_user_id = f"{user_base}_S{synthetic_index}"

        while new_user_id in existing_ids:
            synthetic_index += 1
            new_user_id = f"{user_base}_S{synthetic_index}"

        synthetic_user_map[user_base] = synthetic_index
        existing_ids.add(new_user_id)

        # Augment posts
        new_posts = source_posts.copy()
        new_posts[id_col] = new_user_id
        new_posts[text_col] = new_posts[text_col].apply(augment_text)

        synthetic_users.append(new_posts)

        i += 1
        if i % 10 == 0:
            print(f"➡ Created {i} synthetic users so far...")

    print(f"✅ Total synthetic users created: {len(synthetic_users)}")

    # Combine final dataset
    balanced_df = pd.concat([df] + synthetic_users, ignore_index=True)

    # Shuffle to avoid order bias
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_df


In [24]:
# ---------------------------
# Apply on Train Only
# ---------------------------
df_train_balanced = balance_users(df_train)

print("\nBefore balancing:")
print(df_train.groupby("label")["ID"].nunique())

print("\nAfter balancing:")
print(df_train_balanced.groupby("label")["ID"].nunique())


🟢 Positive users: 20
🔴 Negative users: 132
📌 Target user count: 132
➡ Created 10 synthetic users so far...
➡ Created 20 synthetic users so far...
➡ Created 30 synthetic users so far...
➡ Created 40 synthetic users so far...
➡ Created 50 synthetic users so far...
➡ Created 60 synthetic users so far...
➡ Created 70 synthetic users so far...
➡ Created 80 synthetic users so far...
➡ Created 90 synthetic users so far...
➡ Created 100 synthetic users so far...
➡ Created 110 synthetic users so far...
✅ Total synthetic users created: 112

Before balancing:
label
0    132
1     20
Name: ID, dtype: int64

After balancing:
label
0    132
1    132
Name: ID, dtype: int64


In [26]:
df_train_balanced.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128591 entries, 0 to 128590
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   ID           128591 non-null  object
 1   TITLE        128591 non-null  object
 2   DATE         128591 non-null  object
 3   INFO         128591 non-null  object
 4   TEXT         128591 non-null  object
 5   label        128591 non-null  int64 
 6   TEXT_MERGED  128591 non-null  object
dtypes: int64(1), object(6)
memory usage: 6.9+ MB


In [27]:
df_train_balanced['INFO'].nunique()

1

In [28]:
#save dataframe 
df_train_balanced.to_csv("/kaggle/working/df_train_balancedERisk19.csv", index=False)

In [29]:
df_test.to_csv("/kaggle/working/df_test_PP_ERisk19.csv", index=False)

In [30]:
df_val.to_csv("/kaggle/working/df_val_PP_ERisk19.csv", index=False)

In [1]:
set(df_train_balanced['ID'])

NameError: name 'df_train_balanced' is not defined